# Example 1.2.2 / 5: Checking Computed Values

Companion notebook for **1.2.2 Computed Parameters** in the AMPL lab Documentation, section "Checking Computed Values". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Checking Computed Values

A computed parameter can also have restrictions; they are checked every time the value is
used. Separate them from the formula with a comma. Here `to_pandas()` uses the values, and
raises a `RuntimeError`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                    # products
param frac;                  # fraction of the market kept as inventory; no sign check
param market {PROD} >= 0;    # most that can be sold, in tons
param mininv {p in PROD} = frac * market[p], >= 0;   # checked each time it is used

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["frac"] = -0.1
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # run.py
    print(ampl.param["mininv"].to_pandas())
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: Error executing "display" command:
error processing param mininv['bands']:
	failed check: param mininv['bands'] = -600
		is not >= 0;


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param frac := -0.1;
param market := bands 6000  coils 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
display mininv;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "display" command:
	error processing param mininv['bands']:
		failed check: param mininv['bands'] = -600
			is not >= 0;
[ERROR] 
	Error executing "display" command:
	error processing param mininv['coils']:
		failed check: param mininv['coils'] = -400
			is not >= 0;


mininv [*] :=
bands  -600
coils  -400
;

